# ngbem for Low-Frequency Electromagnetics: Stabilized EFIE

**Date**: February 2026
**Author**: Kengo Sugahara
**Context**: Follow-up to Lucy Weggler's [Maxwell_DtN_Stabilized.ipynb](https://github.com/Weggler/docu-ngsbem/blob/main/demos/Maxwell_DtN_Stabilized.ipynb)

---

## Overview

The stabilized EFIE formulation by Weggler (`HDivSurface` $\times$ `SurfaceL2`
product space) naturally separates the solenoidal and irrotational components
of the surface current — mathematically equivalent to a Loop-Star decomposition,
but achieved elegantly at the variational level without explicit basis construction.

This notebook demonstrates three applications of ngsolve + ngsbem:

1. **Condition number comparison**: $O(1)$ vs $O(\kappa^{-2})$ conditioning
   as $\kappa \to 0$, confirming the low-frequency stability
2. **Air-core coil inductance**: `LaplaceSL` on `HDivSurface` with Hodge
   decomposition to extract the harmonic (circulating) mode from a genus-1
   ring mesh — BEM matches the analytical formula within 0.3%
3. **Calderon preconditioner for EFIE**: Operator preconditioning via the
   rotated single layer operator in the dual sequence (Andriulli et al. /
   Schoeberl), dramatically improving GMRES convergence

Our next goal is to model **coils surrounded by magnetic or conductive bodies**
(e.g., ferrite cores, metallic shields), where the inductance and loss change
due to the penetrable material. The recent low-frequency stabilization of the
PMCHWT equation by Giunzioni et al. [3] appears to be a promising approach
for this, as it handles both dielectric and conductive media with $O(1)$
conditioning in the eddy-current regime.

In [1]:
# === Common imports and utility functions ===
import numpy as np

# Physical constants
MU_0 = 4.0 * np.pi * 1e-7   # H/m


def extract_dense_matrix(mat, ndof):
    """Extract dense numpy matrix from NGSolve BaseMatrix (square)."""
    ei = mat.CreateColVector()
    col = mat.CreateColVector()
    ei[:] = 0; ei[0] = 1.0
    mat.Mult(ei, col)
    is_complex = isinstance(col[0], complex)

    M = np.zeros((ndof, ndof), dtype=complex if is_complex else float)
    for j in range(ndof):
        M[j, 0] = col[j]
    for i in range(1, ndof):
        ei[:] = 0; ei[i] = 1.0
        mat.Mult(ei, col)
        for j in range(ndof):
            M[j, i] = col[j]
    return M


def extract_rect_matrix(mat, nrow, ncol):
    """Extract dense numpy matrix from NGSolve BaseMatrix (rectangular)."""
    ei = mat.CreateRowVector()
    col = mat.CreateColVector()
    M = np.zeros((nrow, ncol))
    for i in range(ncol):
        ei[:] = 0; ei[i] = 1.0
        mat.Mult(ei, col)
        for j in range(nrow):
            M[j, i] = col[j]
    return M


def create_plate_mesh(width, height, maxh, label="conductor"):
    """Create rectangular plate surface mesh using Netgen OCC."""
    from netgen.occ import WorkPlane, OCCGeometry, Axes, Pnt, Dir
    from ngsolve import Mesh
    wp = WorkPlane(Axes(p=Pnt(0, 0, 0), n=Dir(0, 0, 1), h=Dir(1, 0, 0)))
    rect = wp.Rectangle(width, height).Face()
    rect.name = label
    return Mesh(OCCGeometry(rect).GenerateMesh(maxh=maxh))


print("Utilities loaded.")

Utilities loaded.


## 1. Product-Space Formulation and Low-Frequency Stability

The `HDivSurface` $\times$ `SurfaceL2` product space naturally separates the
surface current into solenoidal and irrotational components:

| Component | Function space | Physical meaning |
|:----------|:---------------|:-----------------|
| Solenoidal (div-free) | `HDivSurface` | Loop currents $\to$ inductance ($A_\kappa$) |
| Irrotational | `SurfaceL2` (via `div`) | Charge/potential ($V_\kappa$) |
| Coupling | `div(HDivSurface)` $\times$ `SurfaceL2` | $Q_\kappa$ |

This is mathematically equivalent to the Loop-Star decomposition, but your
formulation achieves the separation at the variational level — no explicit
basis construction is required.

In the classical EFIE, the irrotational part appears as $V_\kappa / \kappa^2$,
which blows up as $\kappa \to 0$. The stabilized formulation uses
$\kappa^2 V_\kappa$ instead, giving $O(1)$ conditioning at all frequencies.

The verification below confirms this.

In [2]:
# === Section 1 verification: Helmholtz-to-Laplace convergence ===

from ngsolve import HDivSurface, SurfaceL2, TaskManager, ds, div
from ngsolve.bem import HelmholtzSL, LaplaceSL

# Create a small mesh for the condition number comparison
mesh_s1 = create_plate_mesh(0.01, 0.01, 0.003, label="conductor")

# Complex function spaces for Helmholtz
fes_hdiv = HDivSurface(mesh_s1, order=0, complex=True)
fes_l2 = SurfaceL2(mesh_s1, order=0, complex=True, dual_mapping=True)
fes_prod = fes_hdiv * fes_l2
(uHDiv, uL2), (vHDiv, vL2) = fes_prod.TnT()

n_loop = fes_hdiv.ndof
n_star = fes_l2.ndof
n_total = fes_prod.ndof
intorder = 8

print(f"Mesh: {n_star} surface elements")
print(f"Product space: {n_loop} loop + {n_star} star = {n_total} DOFs")

# A) Laplace reference (kappa=0)
print("\n--- A) Laplace SL reference (kappa=0) ---")
u_h, v_h = fes_hdiv.TnT()
with TaskManager():
    L_op = LaplaceSL(
        u_h.Trace() * ds(bonus_intorder=intorder)
    ) * v_h.Trace() * ds(bonus_intorder=intorder)

L_laplace = MU_0 * np.real(extract_dense_matrix(L_op.mat, n_loop))
print(f"  L_laplace: ({n_loop}x{n_loop})")

# B) Helmholtz at decreasing kappa
kappa_values = [1.0, 0.1, 0.01, 0.001]

print(f"\n{'kappa':>10s}  {'||L_helm-L_lap||/||L_lap||':>28s}  {'cond(stabilized)':>18s}  {'cond(classical)':>17s}")
print("-" * 80)

for kappa in kappa_values:
    with TaskManager():
        A_k = HelmholtzSL(
            uHDiv.Trace() * ds(bonus_intorder=intorder), kappa
        ) * vHDiv.Trace() * ds(bonus_intorder=intorder)

        V_k = HelmholtzSL(
            uL2 * ds(bonus_intorder=intorder), kappa
        ) * vL2 * ds(bonus_intorder=intorder)

        Q_k = HelmholtzSL(
            div(uHDiv.Trace()) * ds(bonus_intorder=intorder), kappa
        ) * vL2 * ds(bonus_intorder=intorder)

    # Stabilized system
    lhs_stab = A_k.mat + Q_k.mat + Q_k.mat.T + kappa**2 * V_k.mat
    M_stab = extract_dense_matrix(lhs_stab, n_total)

    # Extract A_kappa block -> compare with Laplace
    A_block = M_stab[:n_loop, :n_loop]
    L_helm = MU_0 * np.real(A_block)
    L_diff = np.linalg.norm(L_laplace - L_helm) / np.linalg.norm(L_laplace)

    # Stabilized condition number
    s_stab = np.linalg.svd(M_stab, compute_uv=False)
    s_nz = np.abs(s_stab[np.abs(s_stab) > 1e-14 * np.abs(s_stab[0])])
    cond_stab = s_nz[0] / s_nz[-1]

    # Classical EFIE condition number
    with TaskManager():
        u_cl, v_cl = fes_hdiv.TnT()
        V1 = HelmholtzSL(
            u_cl.Trace() * ds(bonus_intorder=intorder), kappa
        ) * v_cl.Trace() * ds(bonus_intorder=intorder)
        V2 = HelmholtzSL(
            div(u_cl.Trace()) * ds(bonus_intorder=intorder), kappa
        ) * div(v_cl.Trace()) * ds(bonus_intorder=intorder)
        lhs_class = V1.mat - (1.0 / (kappa**2)) * V2.mat

    M_class = extract_dense_matrix(lhs_class, n_loop)
    s_class = np.linalg.svd(M_class, compute_uv=False)
    s_cl_nz = np.abs(s_class[np.abs(s_class) > 1e-14 * np.abs(s_class[0])])
    cond_class = s_cl_nz[0] / s_cl_nz[-1]

    print(f"{kappa:10.4f}  {L_diff:28.6e}  {cond_stab:18.3e}  {cond_class:17.3e}")

print("\nConclusion:")
print("  - Helmholtz A_kappa -> Laplace L as kappa -> 0 (L matrix converges)")
print("  - Stabilized cond stays bounded; classical grows as O(kappa^{-2})")
print("  => The product-space structure provides O(1) conditioning at all kappa")

Mesh: 24 surface elements
Product space: 42 loop + 24 star = 66 DOFs

--- A) Laplace SL reference (kappa=0) ---
  L_laplace: (42x42)

     kappa    ||L_helm-L_lap||/||L_lap||    cond(stabilized)    cond(classical)
--------------------------------------------------------------------------------
    1.0000                  6.757603e-06           2.732e+06          3.129e+06
    0.1000                  6.757632e-08           1.861e+06          3.129e+08
    0.0100                  6.757632e-10           1.854e+06          3.129e+10
    0.0010                  6.757615e-12           1.854e+06          3.129e+12

Conclusion:
  - Helmholtz A_kappa -> Laplace L as kappa -> 0 (L matrix converges)
  - Stabilized cond stays bounded; classical grows as O(kappa^{-2})
  => The product-space structure provides O(1) conditioning at all kappa


    0.0100                  6.757631e-10           1.854e+06          3.129e+10


    0.0010                  6.757584e-12           1.854e+06          3.128e+12

Conclusion:
  - Helmholtz A_kappa -> Laplace L as kappa -> 0 (L matrix converges)
  - Stabilized cond stays bounded; classical grows as O(kappa^{-2})
  => The product-space structure provides O(1) conditioning at all kappa


## 2. Air-Core Coil Inductance from LaplaceSL

As a practical application, we compute the self-inductance of a circular
ring coil using `LaplaceSL` on `HDivSurface` — the $\kappa = 0$ limit of
the $A_\kappa$ block shown in Section 1.

For a genus-1 surface (ring), `HDivSurface` contains a 1D **harmonic
subspace** (divergence-free and curl-free) representing pure circulation
around the hole. We extract this mode via Hodge decomposition and
project the `LaplaceSL` matrix onto it.

Analytical reference for a flat circular loop with trace width $w$:

$$L = \mu_0 R \left[ \ln\!\left(\frac{8R}{w}\right) - \frac{1}{2} \right]$$

In [3]:
# === Section 2: Air-core coil inductance ===

from scipy.linalg import null_space
from ngsolve import BilinearForm, InnerProduct


def create_ring_mesh(R_center, trace_width, maxh, label="conductor"):
    """Annular ring surface mesh (genus-1)."""
    from netgen.occ import WorkPlane, OCCGeometry, Axes, Pnt, Dir
    from ngsolve import Mesh
    R_out, R_in = R_center + trace_width/2, R_center - trace_width/2
    wp = WorkPlane(Axes(Pnt(0, 0, 0), Dir(0, 0, 1), Dir(1, 0, 0)))
    outer = wp.Circle(R_out).Face()
    wp2 = WorkPlane(Axes(Pnt(0, 0, 0), Dir(0, 0, 1), Dir(1, 0, 0)))
    inner = wp2.Circle(R_in).Face()
    ring = outer - inner
    ring.faces.name = label
    return Mesh(OCCGeometry(ring).GenerateMesh(maxh=maxh))


def loop_inductance(mesh, R_center, trace_width, label="conductor"):
    """Self-inductance of genus-1 loop via LaplaceSL + Hodge decomposition."""
    fes_J  = HDivSurface(mesh, order=0)
    fes_L2 = SurfaceL2(mesh, order=0)
    n_J, n_f, n_v = fes_J.ndof, fes_L2.ndof, mesh.nv

    # D: divergence matrix (natural map HDivSurface -> SurfaceL2)
    u_J, q = fes_J.TrialFunction(), fes_L2.TestFunction()
    bf_D = BilinearForm(trialspace=fes_J, testspace=fes_L2)
    bf_D += div(u_J.Trace()) * q * ds
    bf_D.Assemble()
    D = extract_rect_matrix(bf_D.mat, n_f, n_J)

    # C: vertex-edge incidence (graph curl constraint)
    C = np.zeros((n_J, n_v))
    for i, e in enumerate(mesh.edges):
        v = list(e.vertices)
        C[i, v[0].nr], C[i, v[1].nr] = -1, +1

    # M_J: edge mass matrix
    u, v = fes_J.TnT()
    bf_M = BilinearForm(fes_J)
    bf_M += InnerProduct(u.Trace(), v.Trace()) * ds
    bf_M.Assemble()
    M_J = np.real(extract_dense_matrix(bf_M.mat, n_J))

    # Harmonic mode: ker([D; C^T M_J]) -- 1D for genus-1 surface
    c_h = null_space(np.vstack([D, C.T @ M_J]), rcond=1e-10)[:, 0]

    # LaplaceSL on HDivSurface (= kappa->0 limit of A_kappa from Section 1)
    jt, jv = fes_J.TnT()
    with TaskManager():
        V_op = LaplaceSL(
            jt.Trace() * ds(label)
        ) * jv.Trace() * ds(label)
    V_A = np.real(extract_dense_matrix(V_op.mat, n_J))

    # L = mu_0 * <c_h, V_A c_h> * perimeter / (w * <c_h, M_J c_h>)
    #   (normalization-independent: conductivity/thickness cancel in L/R ratio)
    L = MU_0 * (c_h @ V_A @ c_h) * (2 * np.pi * R_center) \
        / (trace_width * (c_h @ M_J @ c_h))
    return L, n_f, n_J


# --- Convergence study ---
R, w = 10e-3, 1e-3   # R = 10 mm, w = 1 mm
L_ref = MU_0 * R * (np.log(8 * R / w) - 0.5)

print(f"Circular loop: R = {R*1e3:.0f} mm, w = {w*1e3:.0f} mm")
print(f"Analytical: L = mu_0*R*[ln(8R/w) - 1/2] = {L_ref*1e9:.2f} nH")
print()

header = "   maxh   faces   edges    L_BEM [nH]    error"
print(header)
print("  " + "-" * 44)

for maxh in [2.0e-3, 1.5e-3, 1.0e-3]:
    mesh_ring = create_ring_mesh(R, w, maxh)
    L_bem, nf, nj = loop_inductance(mesh_ring, R, w)
    err = abs(L_bem - L_ref) / L_ref * 100
    print(f"  {maxh*1e3:6.1f}mm  {nf:6d}  {nj:6d}  {L_bem*1e9:10.2f}  {err:6.1f}%")

print(f"\nBEM = {L_bem*1e9:.2f} nH  vs  Analytical = {L_ref*1e9:.2f} nH")

Circular loop: R = 10 mm, w = 1 mm
Analytical: L = mu_0*R*[ln(8R/w) - 1/2] = 48.78 nH

   maxh   faces   edges    L_BEM [nH]    error
  --------------------------------------------
     2.0mm      63     126       48.67     0.2%
     1.5mm      84     168       48.63     0.3%
     1.0mm     126     252       48.63     0.3%

BEM = 48.63 nH  vs  Analytical = 48.78 nH


## 3. Calderon Preconditioner for EFIE (Andriulli / Schoeberl)

While the product-space reformulation in Section 1 achieves $O(1)$ conditioning
by restructuring the block system, an alternative approach is **operator
preconditioning**: apply a Calderon-type preconditioner to the standard EFIE.

The key idea (Andriulli et al. [4]) is to precondition the single layer
operator $\mathcal{S}$ by its **rotated** counterpart $(\cdot \times n)
\circ \mathcal{S} \circ (\cdot \times n)$, which maps `HDivSurface` to itself
via the dual sequence.

The preconditioner has the structure:

$$P = M^{-1} \left( \kappa \, V_{\mathrm{rot}} - \frac{1}{\kappa} \nabla_s \, M_{H^1}^{-1} \, V_{\mathrm{pot}} \, M_{H^1}^{-1} \, \nabla_s^T \right) M^{-1}$$

where:
- $V_{\mathrm{rot}}$: `HelmholtzSL` applied to rotated basis $(n \times u)$
- $V_{\mathrm{pot}}$: `HelmholtzSL` on scalar `H1` space
- $\nabla_s$: surface curl operator
- $M, M_{H^1}$: mass matrices on `HDivSurface` and `H1`

This requires two function spaces (`HDivSurface` + `H1`) but works with
the **standard EFIE** — no product-space reformulation needed.

**Implementation below is adapted from Joachim Schoeberl's `EMpre.ipynb` (2026-02).**

In [4]:
# === Section 3: Calderon preconditioner for EFIE ===
# Adapted from Joachim Schoeberl's Maxwell_EFIE_Preconditioner.ipynb (2026-02)

from ngsolve import *
from netgen.occ import *
from ngsolve.bem import *
from time import time

# --- Sphere mesh ---
face = Glue(Sphere((0,0,0), 1).faces)
mesh_s3 = Mesh(OCCGeometry(face).GenerateMesh(maxh=0.3)).Curve(4)

# --- EFIE setup ---
kappa = 3*pi
d = CF((0, -kappa, 0))
E_inc = exp(1j * d * CF((x,y,z))) * CF((1, 0, 0))

fes_s3 = HDivSurface(mesh_s3, order=2, complex=True)
u3, v3 = fes_s3.TnT()

fespot_s3 = H1(mesh_s3, order=3, complex=True)
upot3, vpot3 = fespot_s3.TnT()

print(f"Sphere mesh: HDivSurface ndof = {fes_s3.ndof}, H1 ndof = {fespot_s3.ndof}")

# --- EFIE operator ---
print("\nAssembling EFIE operator...")
t0 = time()
with TaskManager():
    V1_s3 = HelmholtzSL(
        u3.Trace()*ds(bonus_intorder=4), kappa
    ) * v3.Trace()*ds(bonus_intorder=4)
    V2_s3 = HelmholtzSL(
        div(u3.Trace())*ds(bonus_intorder=4), kappa
    ) * div(v3.Trace())*ds(bonus_intorder=4)
print(f"  EFIE assembly: {time()-t0:.1f}s")

lhs_s3 = (1j*kappa) * V1_s3.mat + 1/(1j*kappa) * V2_s3.mat
rhs_s3 = LinearForm(E_inc*v3.Trace()*ds(bonus_intorder=3)).Assemble()

# --- Calderon preconditioner ---
print("Assembling Calderon preconditioner...")
t0 = time()
invMHd = BilinearForm(u3.Trace()*v3.Trace()*ds).Assemble().mat.Inverse(inverse="sparsecholesky")
n_s3 = specialcf.normal(3)

with TaskManager():
    Vrot = HelmholtzSL(
        Cross(u3.Trace(), n_s3)*ds(bonus_intorder=2), kappa
    ) * Cross(v3.Trace(), n_s3)*ds(bonus_intorder=2)
    Vpot = HelmholtzSL(
        upot3*ds(bonus_intorder=2), kappa
    ) * vpot3*ds(bonus_intorder=2)

surfcurl = BilinearForm(
    Cross(grad(upot3).Trace(), n_s3) * v3.Trace()*ds
).Assemble().mat
invMH1 = BilinearForm(upot3*vpot3*ds).Assemble().mat.Inverse(inverse="sparsecholesky")

pre_s3 = invMHd @ (
    kappa*Vrot.mat
    - 1/kappa*surfcurl@invMH1@Vpot.mat@invMH1@surfcurl.T
) @ invMHd
print(f"  Preconditioner assembly: {time()-t0:.1f}s")

# --- GMRES with preconditioner ---
print("\n--- GMRES with Calderon preconditioner ---")
gfj_pre = GridFunction(fes_s3)
t0 = time()
with TaskManager():
    gfj_pre.vec[:] = solvers.GMRes(A=lhs_s3, b=rhs_s3.vec, pre=pre_s3,
                                    maxsteps=500, tol=1e-8, printrates=False)
t_pre = time() - t0
print(f"  Solve time: {t_pre:.1f}s")

# --- GMRES without preconditioner (mass inverse as trivial pre) ---
print("\n--- GMRES without preconditioner ---")
pre_id = BilinearForm(u3.Trace()*v3.Trace()*ds).Assemble().mat.Inverse(inverse="sparsecholesky")
gfj_nopre = GridFunction(fes_s3)
t0 = time()
with TaskManager():
    gfj_nopre.vec[:] = solvers.GMRes(A=lhs_s3, b=rhs_s3.vec, pre=pre_id,
                                      maxsteps=500, tol=1e-8, printrates=False)
t_nopre = time() - t0
print(f"  Solve time: {t_nopre:.1f}s")

# --- Compare ---
diff = gfj_pre.vec.CreateVector()
diff.data = gfj_pre.vec - gfj_nopre.vec
rel_diff = diff.Norm() / gfj_pre.vec.Norm()
print(f"\nRelative difference between solutions: {rel_diff:.2e}")
print(f"Speedup: {t_nopre/t_pre:.1f}x")


Sphere mesh: HDivSurface ndof = 2310, H1 ndof = 1388

Assembling EFIE operator...
  EFIE assembly: 22.7s
Assembling Calderon preconditioner...
  Preconditioner assembly: 11.3s

--- GMRES with Calderon preconditioner ---
  Solve time: 137.4s

--- GMRES without preconditioner ---
  Solve time: 695.8s

Relative difference between solutions: 6.09e-09
Speedup: 5.1x


  Solve time: 35.0s

--- GMRES without preconditioner ---


  Solve time: 177.7s

Relative difference between solutions: 6.09e-09
Speedup: 5.1x


## Prerequisites

This notebook is **fully self-contained** — no external Python modules are needed.

```bash
pip install ngsolve==6.2.2405
pip install ngsolve-ngsbem      # or install from source
pip install numpy scipy
```

### What this notebook uses from ngsbem

| ngsbem operator | Purpose |
|:----------------|:--------|
| `HelmholtzSL(HDivSurface)` | Vector SLP $A_\kappa$ (Sections 1, 3) |
| `HelmholtzSL(SurfaceL2)` | Scalar SLP $V_\kappa$ for stabilized block (Section 1) |
| `HelmholtzSL(div(HDivSurface), SurfaceL2)` | Coupling $Q_\kappa$ (Section 1) |
| `HelmholtzSL(Cross(HDivSurface, n))` | Rotated SLP $V_\mathrm{rot}$ for Calderon preconditioner (Section 3) |
| `HelmholtzSL(H1)` | Scalar potential SLP $V_\mathrm{pot}$ for preconditioner (Section 3) |
| `LaplaceSL(HDivSurface)` | Inductance matrix ($\kappa = 0$ limit) (Sections 1 & 2) |

Section 2 additionally uses `scipy.linalg.null_space` for the Hodge decomposition.

## Summary

| What | Significance |
|:---|:---|
| `HDivSurface` $\times$ `SurfaceL2` product space | Natural Loop-Star separation without explicit basis construction |
| $O(1)$ condition number for all $\kappa$ | Low-frequency (MQS) limit is inherently stable |
| $A_\kappa$ block $\to$ Laplace $L$ as $\kappa \to 0$ | Laplace BEM = correct inductance matrix |
| `LaplaceSL` on genus-1 `HDivSurface` | Air-core coil inductance via Hodge decomposition (0.3% error) |
| Calderon preconditioner via rotated SL | Dramatic GMRES speedup for standard EFIE (Section 3) |

### Results

1. **Condition number** (Section 1): The stabilized formulation maintains $O(1)$ conditioning
   as $\kappa \to 0$, while the classical EFIE grows as $O(\kappa^{-2})$.

2. **Air-core inductance** (Section 2): `LaplaceSL` projected onto the harmonic subspace
   of `HDivSurface` gives accurate self-inductance for a circular ring coil
   (BEM $\approx$ 48.7 nH vs analytical 48.8 nH for $R = 10$ mm, $w = 1$ mm).

3. **Calderon preconditioner** (Section 3): The rotated-SL preconditioner by
   Schoeberl (based on Andriulli et al. [4]) significantly accelerates GMRES
   convergence for the standard EFIE on `HDivSurface`. This is complementary
   to the product-space stabilization: the product space restructures the system,
   while the Calderon preconditioner accelerates the iterative solver.

### Two Approaches to Low-Frequency EFIE

| Approach | Mechanism | Strength |
|:---------|:----------|:---------|
| Product-space (Weggler) | Reformulation into $[A_\kappa, Q_\kappa; Q_\kappa^T, \kappa^2 V_\kappa]$ | $O(1)$ condition number; natural Loop-Star separation |
| Calderon preconditioner (Andriulli/Schoeberl) | Operator preconditioning via $(n \times) \circ \mathcal{S} \circ (n \times)$ | Works with standard EFIE; fast GMRES convergence |

Both achieve stable low-frequency behavior. The product-space approach is
ideal for PEEC-type circuit extraction; the Calderon preconditioner is
more general for full-wave EM scattering.

### References

1. L. Weggler, "Stabilized Maxwell DtN Formulation," ngsbem demo, 2026.
   [Maxwell_DtN_Stabilized.ipynb](https://github.com/Weggler/docu-ngsbem/blob/main/demos/Maxwell_DtN_Stabilized.ipynb)
2. J. Ostrowski et al., "ngbem -- A BEM Library for NGSolve," 2024.
3. V. Giunzioni, A. Scazzola, A. Merlini, and F. P. Andriulli,
   "Low-Frequency Stabilizations of the PMCHWT Equation for Dielectric
   and Conductive Media: On a Full-Wave Alternative to Eddy-Current Solvers,"
   *IEEE Trans. Antennas Propag.*, vol. 73, no. 8, pp. 5725--5740, 2025.
   [arXiv:2408.01321](https://arxiv.org/abs/2408.01321)
4. S. B. Adrian, A. Dely, D. Consoli, A. Merlini, and F. P. Andriulli,
   "Electromagnetic Integral Equations: Insights in Conditioning and Preconditioning,"
   *IEEE Open J. Antennas Propag.*, vol. 2, pp. 1143--1174, 2021.
   [DOI: 10.1109/OJAP.2021.3121097](https://doi.org/10.1109/OJAP.2021.3121097)